In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kacpergregorowicz/house-plant-species") + '\\house_plant_species'

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Школа Рока\.cache\kagglehub\datasets\kacpergregorowicz\house-plant-species\versions\4\house_plant_species


In [14]:
from torchvision.models import resnet18
from AircraftDataset import AircraftDataset
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
from model import PlantCNN

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [16]:
def evaluate_model(model, data_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

In [17]:
import torch
from torch import nn, optim
from tqdm import tqdm


def train_model(model, train_loader, val_loader, device, epochs=10):
    # wandb.init(project="aircraft-classifier",
    #            config={"epochs": epochs,
    #                    "lr": 1e-4,
    #                    "batch_size": train_loader.batch_size}
    #            )
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    best_acc = 0.0
    patience = 3
    counter = 0

    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}"):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = correct / total
        val_acc = evaluate_model(model, val_loader, device)

        # wandb.log({
        #     "train_loss": running_loss / total,
        #     "train_acc": train_acc,
        #     "val_acc": val_acc,
        #     "epoch": epoch + 1
        # })

        if val_acc > best_acc:
            best_acc = val_acc
            counter = 0
            torch.save(model.state_dict(), "best_model.pth")
            print("✅ Model saved.")
            # wandb.run.summary["best_val_acc"] = best_acc
        else:
            counter += 1
            if counter >= patience:
                print('Early stopping triggered')
                break
        scheduler.step()

        print(f"Epoch {epoch + 1}: Loss={running_loss / total:.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

### Загрузим данные

In [18]:
from data_loader import load_data

In [19]:
train_small_dataset, val_small_dataset, test_small_dataset, classes = load_data(path, 0.1, transform) 

In [20]:
train_small = DataLoader(train_small_dataset, batch_size=16, pin_memory=True, num_workers=4, shuffle=True)
val_small = DataLoader(val_small_dataset, batch_size=16, pin_memory=True, num_workers=4)

In [21]:
model = PlantCNN(num_classes=47)
model = model.to(device)

In [22]:
train_model(model, train_small, val_small, device, epochs=20)

Epoch 1/20: 100%|██████████| 74/74 [00:37<00:00,  1.97it/s]


✅ Model saved.
Epoch 1: Loss=3.7869, Train Acc=0.0542, Val Acc=0.1493


Epoch 2/20: 100%|██████████| 74/74 [00:38<00:00,  1.90it/s]


Epoch 2: Loss=3.5744, Train Acc=0.1025, Val Acc=0.1176


Epoch 3/20: 100%|██████████| 74/74 [00:37<00:00,  1.96it/s]


...

Epoch 10: Loss=2.8941, Train Acc=0.2295, Val Acc=0.2534


Epoch 11/20: 100%|██████████| 74/74 [00:36<00:00,  2.04it/s]


Epoch 11: Loss=2.8300, Train Acc=0.2523, Val Acc=0.2896


Epoch 12/20: 100%|██████████| 74/74 [00:34<00:00,  2.13it/s]


Early stopping triggered


### Добавим аугментации

In [23]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [24]:
train_small_dataset, val_small_dataset, test_small_dataset, classes = load_data(path, 0.1, transform) 

train_small_loader = DataLoader(train_small_dataset, batch_size=16, pin_memory=True, num_workers=8, shuffle=True)
val_small_loader = DataLoader(val_small_dataset, batch_size=16, pin_memory=True, num_workers=8)

C:\ProgramData\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [25]:
model = PlantCNN(num_classes=47)
model = model.to(device)

In [26]:
train_model(model, train_small_loader, val_small_loader, device, epochs=20)

Epoch 1/20: 100%|██████████| 74/74 [00:49<00:00,  1.48it/s]


✅ Model saved.
Epoch 1: Loss=3.8086, Train Acc=0.0466, Val Acc=0.0905


Epoch 2/20: 100%|██████████| 74/74 [00:49<00:00,  1.49it/s]


✅ Model saved.
Epoch 2: Loss=3.5811, Train Acc=0.0923, Val Acc=0.1222


Epoch 3/20: 100%|██████████| 74/74 [00:49<00:00,  1.50it/s]


...

Epoch 14: Loss=2.9079, Train Acc=0.2312, Val Acc=0.2760


Epoch 15/20: 100%|██████████| 74/74 [00:50<00:00,  1.48it/s]


Epoch 15: Loss=2.8923, Train Acc=0.2329, Val Acc=0.2624


Epoch 16/20: 100%|██████████| 74/74 [00:50<00:00,  1.47it/s]


Early stopping triggered


### Наблюдается огромное недообучение. Ситуация улучшилась с 2.7% до 3.1%, но этого по-прежнему недостаточно. Попробуем изменить архитектуру модели, возьмём предобученный ResNet18 с изменённым последним слоем под 100 классов

In [27]:
model = resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 47)
model = model.to(device)

In [28]:
train_model(model, train_small_loader, val_small_loader, device, epochs=20)

Epoch 1/20: 100%|██████████| 74/74 [00:50<00:00,  1.46it/s]


✅ Model saved.
Epoch 1: Loss=3.0303, Train Acc=0.2854, Val Acc=0.5158


Epoch 2/20: 100%|██████████| 74/74 [00:51<00:00,  1.44it/s]


✅ Model saved.
Epoch 2: Loss=1.5239, Train Acc=0.6994, Val Acc=0.6787


Epoch 3/20: 100%|██████████| 74/74 [00:51<00:00,  1.45it/s]


...

Epoch 16: Loss=0.0515, Train Acc=1.0000, Val Acc=0.8507


Epoch 17/20: 100%|██████████| 74/74 [00:50<00:00,  1.47it/s]


Epoch 17: Loss=0.0476, Train Acc=1.0000, Val Acc=0.8507


Epoch 18/20: 100%|██████████| 74/74 [00:50<00:00,  1.46it/s]


Early stopping triggered


### Модель обучилась в 10 раз лучше, что говорит о том, что изменение архитектуры дало положительные результаты. Недообучение по-прежнему наблюдается, попробуем расширить данные - добавим больше картинок.

In [29]:
train_dataset, val_dataset, test_dataset, classes = load_data(path, 1, transform) 

train_loader = DataLoader(train_dataset, batch_size=16, pin_memory=True, num_workers=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, pin_memory=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=16, pin_memory=True, num_workers=4)

In [30]:
model = resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 47)
model = model.to(device)

In [31]:
train_model(model, train_loader, val_loader, device, epochs=20)

Epoch 1/20: 100%|██████████| 739/739 [03:32<00:00,  3.48it/s]


✅ Model saved.
Epoch 1: Loss=1.1982, Train Acc=0.7039, Val Acc=0.8569


Epoch 2/20: 100%|██████████| 739/739 [03:34<00:00,  3.44it/s]


✅ Model saved.
Epoch 2: Loss=0.4691, Train Acc=0.8742, Val Acc=0.8899


Epoch 3/20: 100%|██████████| 739/739 [03:30<00:00,  3.51it/s]


...

Epoch 8: Loss=0.0543, Train Acc=0.9879, Val Acc=0.9219


Epoch 9/20: 100%|██████████| 739/739 [03:42<00:00,  3.32it/s]


Epoch 9: Loss=0.0389, Train Acc=0.9927, Val Acc=0.9260


Epoch 10/20: 100%|██████████| 739/739 [03:43<00:00,  3.30it/s]


Early stopping triggered


### Расширенный набор данных (Kaggle + Google) улучшил точность модели до приемлемого уровня, цель достигнута. 